In [37]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-haiku-4-5"

In [38]:
# Helper functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

In [39]:
import json


def generate_dataset():
    prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task",
        "format": "json"
        "solution_criteria": "Key criteria for evaluating the solution"
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single JSON object.
* Focus on tasks that do not require writing much code
* Solution criteria must include runtime, memory size, timeout and basic structure for AWS Lambda configuation.

Please generate 3 objects.
"""
    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences=["```"])
    #text = chat(messages)
    
    return json.loads(text)

In [40]:
dataset = generate_dataset()

dataset

[{'task': 'Create a CloudFormation template JSON to provision an S3 bucket with versioning enabled and public access blocked',
  'format': 'json',
  'solution_criteria': 'Must include Resources section with AWS::S3::Bucket and AWS::S3::PublicAccessBlockConfiguration, VersioningConfiguration property set to Enabled, BlockPublicAcls and BlockPublicPolicy set to true, valid CloudFormation syntax, deployable without errors'},
 {'task': 'Generate a Lambda function configuration JSON for an environment that processes SQS messages with 512MB memory, 60-second timeout, and requires IAM role with SQS read permissions',
  'format': 'json',
  'solution_criteria': 'Must include MemorySize (512), Timeout (60), Environment variables section, Role ARN reference, Handler and Runtime fields, valid Lambda function configuration schema, compatible with CloudFormation or AWS API deployment'},
 {'task': 'Create an AWS EventBridge rule configuration JSON that triggers a Lambda function every 5 minutes with 

In [41]:
with open("dataset.json", "w") as f:
    json.dump(dataset, f, indent=2)

In [42]:

def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""

    prompt = f"""
Please solve the following task:

{test_case["task"]}

* Respond only with Python, JSON, or a plain Regex
* Do not add any comments or commentary or explaination
"""
    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```code")
    output = chat(messages,  stop_sequences=["```"])
    return output



In [43]:
# Function to grade a test case + output using a model
def grade_by_model(test_case, output):
    eval_prompt = f"""
You are an expert AWS code reviewer. Your task is to evaluate the following AI-generated solution.

Original Task:
<task>
{test_case["task"]}
</task>

Solution to Evaluate:
<solution>
{output}
</solution>

Criteria you should use to evaulate the solution:
<criteria>
{test_case["solution_criteria"]}
</criteria>

Output Format
Provide your evaluation as a structured JSON object with the following fields, in this specific order:
- "strengths": An array of 1-3 key strengths
- "weaknesses": An array of 1-3 key areas for improvement
- "reasoning": A concise explanation of your overall assessment
- "score": A number between 1-10

Respond with JSON. Keep your response concise and direct.
Example response shape:
{{
    "strengths": string[],
    "weaknesses": string[],
    "reasoning": string,
    "score": number
}}
    """

    messages = []
    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, "```json")
    eval_text = chat(messages, stop_sequences=["```"])
    return json.loads(eval_text)

In [44]:
# Functions to validate the output structure
import re
import ast


def validate_json(text):
    try:
        json.loads(text.strip())
        return 10
    except json.JSONDecodeError:
        return 0


def validate_python(text):
    try:
        ast.parse(text.strip())
        return 10
    except SyntaxError:
        return 0


def validate_regex(text):
    try:
        re.compile(text.strip())
        return 10
    except re.error:
        return 0


def grade_syntax(response, test_case):
    format = test_case["format"]
    if format == "json":
        return validate_json(response)
    elif format == "python":
        return validate_python(response)
    else:
        return validate_regex(response)

In [45]:
def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)

    # TODO - Grading
    model_grade = grade_by_model(test_case, output)
    model_score = model_grade["score"]
    reasoning = model_grade["reasoning"]

    syntax_score = grade_syntax(output, test_case)

    score = (model_score + syntax_score) / 2

    return {
        "output": output, 
        "test_case": test_case,
        "score": score,
        "reasoning": reasoning
        }  

    

In [46]:
from statistics import mean

def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []

    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

    average_score = mean([result["score"] for result in results])
    print(f"Average score: {average_score}")

    return results

In [47]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)

result = run_eval(dataset)

Average score: 8.5


In [48]:
print(json.dumps(result, indent=2))

[
  {
    "output": "\n{\n  \"AWSTemplateFormatVersion\": \"2010-09-09\",\n  \"Description\": \"CloudFormation template to provision an S3 bucket with versioning enabled and public access blocked\",\n  \"Resources\": {\n    \"S3Bucket\": {\n      \"Type\": \"AWS::S3::Bucket\",\n      \"Properties\": {\n        \"BucketName\": {\n          \"Fn::Sub\": \"my-secure-bucket-${AWS::AccountId}\"\n        },\n        \"VersioningConfiguration\": {\n          \"Status\": \"Enabled\"\n        }\n      }\n    },\n    \"S3BucketPublicAccessBlock\": {\n      \"Type\": \"AWS::S3::PublicAccessBlock\",\n      \"Properties\": {\n        \"BucketName\": {\n          \"Ref\": \"S3Bucket\"\n        },\n        \"BlockPublicAcls\": true,\n        \"BlockPublicPolicy\": true,\n        \"IgnorePublicAcls\": true,\n        \"RestrictPublicBuckets\": true\n      }\n    }\n  },\n  \"Outputs\": {\n    \"BucketName\": {\n      \"Description\": \"Name of the S3 bucket\",\n      \"Value\": {\n        \"Ref\": \"S3